# Demo end-to-end: happy path de trips en Pylondrina

En este notebook voy a probar un flujo simple pero completo del módulo:

**Import trips → validate trips → Write trips → Read trips → validate trips**

La idea es trabajar con un dataset sintético rico, con varias columnas categóricas y suficiente tamaño como para que:
- la validación sea exigente pero exitosa,
- la persistencia formal se vea claramente,
- y el Parquet resultante muestre una ventaja real frente al CSV crudo.

### Configuración inicial

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic"

### 0. Setup e imports

In [ ]:
from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq

from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.io.trips import WriteTripsOptions, ReadTripsOptions, write_trips, read_trips

RAW_CSV_PATH = DATA_PATH / "demo_trips_happy_path_source.csv"
OUTPUT_DIR = DATA_PATH / "demo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_BASE_PATH = OUTPUT_DIR / "demo_trips_happy_path"
ARTIFACT_DIR = OUTPUT_DIR / "demo_trips_happy_path.golondrina"

print("RAW_CSV_PATH =", RAW_CSV_PATH.resolve())
print("ARTIFACT_DIR =", ARTIFACT_DIR.resolve())

RAW_CSV_PATH = D:\Memoria\repos\pylondrina\data\synthetic\demo_trips_happy_path_source.csv
ARTIFACT_DIR = D:\Memoria\repos\pylondrina\data\synthetic\demo_outputs\demo_trips_happy_path.golondrina


## 1. Carga inicial del CSV crudo

Primero voy a cargar el CSV fuente tal como salió del generador.  
Aquí todavía estoy viendo una “fuente externa”: nombres de columnas no canónicos y algunos valores categóricos que luego deberán mapearse.

In [4]:
raw_df = pd.read_csv(RAW_CSV_PATH)

print("Shape crudo:", raw_df.shape)
print("Cantidad de columnas:", len(raw_df.columns))
display(raw_df.head())

display(
    pd.DataFrame({
        "dtype": raw_df.dtypes.astype(str),
        "n_nulls": raw_df.isna().sum(),
    }).head(30)
)

print("Valores únicos de modo_fuente (muestra):")
print(sorted(raw_df["modo_fuente"].dropna().astype(str).unique().tolist())[:30])

print("Valores únicos de proposito_fuente (muestra):")
print(sorted(raw_df["proposito_fuente"].dropna().astype(str).unique().tolist())[:30])

print("Tamaño CSV crudo (bytes):", RAW_CSV_PATH.stat().st_size)

Shape crudo: (10000, 46)
Cantidad de columnas: 46


,id_movimiento,id_usuario,id_viaje,seq_movimiento,lon_origen,lat_origen,lon_destino,lat_destino,fecha_hora_origen,fecha_hora_destino,h3_origen,h3_destino,comuna_origen,comuna_destino,offset_tz_min,hora_local_origen,hora_local_destino,factor_expansion,secuencia_modos,modo_fuente,proposito_fuente,tipo_dia,franja_horaria,genero_usuario,grupo_etario,quintil_ingreso,activity_destination,season,travel_time_quality_code,fare_payment_type,highway_avoid_reason,bike_lane_usage,bike_lane_issue,bike_parking_type,cut_type_stage_trip,travel_time_bucket,home_tenure,education_level,activity_status,occupation_type,travel_time_min,fare_amount,distance_route_m,in_vehicle_travel_time_min,household_id,public_transport_route_code
0,m00000,u1079,tm_00000,0,-70.594233,-33.452177,-70.546872,-33.444157,2026-03-08T01:48:00Z,2026-03-08T02:05:00Z,88b2c50b67fffff,88b2c50b51fffff,Vitacura,Vitacura,-180,01:48,02:05,4.488,train,Furgón escolar chofer,Comer o Tomar algo,holiday,afternoon,male,unknown,unknown,recreación,winter,imputed,cash,no_need,not_applicable,missing_segments,private_building,cut_at_missing_stage,0-10,owned,secondary,retired,employer,17.0,1520.0,10653.7,17.0,H03082,I09c
1,m00001,u1079,tm_00000,1,-70.443241,-33.252172,-70.388813,-33.256286,2026-03-07T09:23:00Z,2026-03-07T10:53:00Z,88b2c5c4d7fffff,88b2c51315fffff,Independencia,Puente Alto,-180,09:23,10:53,2.493,bicycle,ZP,leisure,weekend,morning,male,65-plus,unknown,salud,winter,missing_end,cash,too_expensive,never,unsafe,private_building,single_stage_only,41-60,owned,technical,retired,employee,90.0,1520.0,1357.5,90.0,H04605,L4
2,m00002,u1637,tm_00001,0,-70.681063,-33.341502,-70.705876,-33.341132,2026-03-05T04:40:00Z,2026-03-05T05:28:00Z,88b2c5cd8bfffff,88b2c5cda9fffff,Ñuñoa,Santiago,-180,04:40,05:28,3.416,bus,METROTREN,Al estudio,weekday,night,unknown,15-24,2,estudio,vacation,missing_end,card,too_expensive,not_applicable,missing_segments,none,single_stage_only,41-60,rented,technical,other,employee,48.0,800.0,1707.4,48.0,H04806,I09c
3,m00003,u1637,tm_00001,1,-70.557340,-33.481444,-70.565914,-33.449085,2026-03-04T10:41:00Z,2026-03-04T12:19:00Z,88b2c5086dfffff,88b2c50b09fffff,Pudahuel,Providencia,-180,10:41,12:19,4.030,car+car+car,Furgón escolar pasajero,work,weekday,midday,male,15-24,3,recreación,vacation,reported_approx,card,no_need,not_applicable,crowded,none,cut_at_missing_stage,0-10,other,primary,studying,employee,98.0,800.0,3429.2,98.0,H03436,B07
4,m00004,u1637,tm_00001,2,-70.695597,-33.513595,-70.658855,-33.509041,2026-03-03T13:22:00Z,2026-03-03T14:08:00Z,88b2c54665fffff,88b2c54609fffff,Independencia,La Florida,-180,13:22,14:08,1.746,car+walk,Auto Chofer,Visitar a alguien,weekday,afternoon,female,0-14,3,estudio,vacation,missing_end,integrated_fare,prefers_local_roads,always,missing_segments,private_building,cut_at_missing_stage,0-10,loaned,primary,retired,employer,46.0,1200.0,8659.5,46.0,H04054,401


,dtype,n_nulls
id_movimiento,object,0
id_usuario,object,0
id_viaje,object,0
seq_movimiento,int64,0
lon_origen,float64,0
lat_origen,float64,0
lon_destino,float64,0
lat_destino,float64,0
fecha_hora_origen,object,0
fecha_hora_destino,object,0


Valores únicos de modo_fuente (muestra):
['Auto Acompañante', 'Auto Chofer', 'Bus alimentador', 'Bus institucional', 'Bus interurbano o rural', 'Bus troncal', 'Bus urbano pago conductor', 'Furgón escolar chofer', 'Furgón escolar pasajero', 'METROTREN', 'Servicio informal', 'Taxi colectivo', 'Taxi o radiotaxi', 'ZP', 'bicycle', 'bus', 'car', 'metro', 'motorcycle', 'other', 'ride_hailing', 'scooter', 'taxi', 'train', 'walk']
Valores únicos de proposito_fuente (muestra):
['Al estudio', 'Al trabajo', 'Buscar o Dejar a alguien', 'Buscar o dejar algo', 'Comer o Tomar algo', 'De compras', 'Otra actividad', 'Por estudio', 'Por trabajo', 'Recreación', 'Trámites', 'Visitar a alguien', 'education', 'errand', 'health', 'home', 'leisure', 'other', 'shopping', 'transfer', 'volver a casa', 'work']
Tamaño CSV crudo (bytes): 4577737


## 2. Definición del contrato de importación

Ahora voy a construir el `TripSchema` que usaré para el happy path.

La idea es que:
- los campos requeridos queden bien definidos,
- los dominios categóricos sean explícitos,
- y además incluya varias columnas opcionales y extendidas para que el dataset final sea más rico y el Parquet tenga más oportunidades de optimizar categorías.

In [8]:
SOURCE_FIELD_CORRESPONDENCE = {
    "movement_id": "id_movimiento",
    "user_id": "id_usuario",
    "origin_longitude": "lon_origen",
    "origin_latitude": "lat_origen",
    "destination_longitude": "lon_destino",
    "destination_latitude": "lat_destino",
    "origin_h3_index": "h3_origen",
    "destination_h3_index": "h3_destino",
    "origin_time_utc": "fecha_hora_origen",
    "destination_time_utc": "fecha_hora_destino",
    "trip_id": "id_viaje",
    "movement_seq": "seq_movimiento",
    "origin_municipality": "comuna_origen",
    "destination_municipality": "comuna_destino",
    "timezone_offset_min": "offset_tz_min",
    "origin_time_local_hhmm": "hora_local_origen",
    "destination_time_local_hhmm": "hora_local_destino",
    "trip_weight": "factor_expansion",
    "mode_sequence": "secuencia_modos",
    "mode": "modo_fuente",
    "purpose": "proposito_fuente",
    "day_type": "tipo_dia",
    "time_period": "franja_horaria",
    "user_gender": "genero_usuario",
    "user_age_group": "grupo_etario",
    "income_quintile": "quintil_ingreso",
}

VALUE_CORRESPONDENCE = {
    "mode": {
        "Auto Chofer": "car",
        "Auto Acompañante": "car",
        "Bus alimentador": "bus",
        "Bus troncal": "bus",
        "Bus institucional": "bus",
        "Bus interurbano o rural": "bus",
        "Bus urbano pago conductor": "bus",
        "Taxi colectivo": "taxi",
        "Taxi o radiotaxi": "taxi",
        "Furgón escolar pasajero": "bus",
        "Furgón escolar chofer": "bus",
        "Servicio informal": "other",
        "METROTREN": "train",
        "ZP": "bus",
    },
    "purpose": {
        "Al trabajo": "work",
        "Por trabajo": "work",
        "Al estudio": "education",
        "Por estudio": "education",
        "volver a casa": "home",
        "Visitar a alguien": "leisure",
        "Buscar o Dejar a alguien": "errand",
        "Buscar o dejar algo": "errand",
        "Comer o Tomar algo": "leisure",
        "De compras": "shopping",
        "Trámites": "errand",
        "Recreación": "leisure",
        "Otra actividad": "other",
    },
}

CANONICAL_DOMAINS = {
    "mode": [
        "walk", "bicycle", "scooter", "motorcycle", "car",
        "taxi", "ride_hailing", "bus", "metro", "train", "other",
    ],
    "purpose": [
        "home", "work", "education", "shopping", "errand",
        "health", "leisure", "transfer", "other",
    ],
    "day_type": ["weekday", "weekend", "holiday"],
    "time_period": ["night", "morning", "midday", "afternoon", "evening"],
    "user_gender": ["female", "male", "other", "unknown"],
    "user_age_group": ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
    "income_quintile": ["1", "2", "3", "4", "5", "unknown"],
}

EXTRA_CATEGORICAL_DOMAINS = {
    "activity_destination": ["trabajo", "estudio", "compra", "salud", "recreación"],
    "season": ["summer", "winter", "normal_period", "vacation"],
    "travel_time_quality_code": ["reported_exact", "reported_approx", "imputed", "missing_end"],
    "fare_payment_type": ["cash", "card", "integrated_fare", "free_transfer", "not_applicable"],
    "highway_avoid_reason": ["too_expensive", "traffic", "no_need", "prefers_local_roads"],
    "bike_lane_usage": ["always", "sometimes", "never", "not_applicable"],
    "bike_lane_issue": ["missing_segments", "unsafe", "poor_maintenance", "crowded", "none"],
    "bike_parking_type": ["none", "street_rack", "guarded", "private_building"],
    "cut_type_stage_trip": ["complete_trip", "partial_trip", "cut_at_missing_stage", "single_stage_only"],
    "travel_time_bucket": ["0-10", "11-20", "21-40", "41-60", "60+"],
    "home_tenure": ["owned", "rented", "loaned", "other"],
    "education_level": ["none", "primary", "secondary", "technical", "university", "postgraduate"],
    "activity_status": ["working", "studying", "homemaker", "retired", "unemployed", "other"],
    "occupation_type": ["employee", "self_employed", "employer", "public_sector", "informal"],
}

def issues_to_frame(issues):
    if not issues:
        return pd.DataFrame()

    rows = []
    for issue in issues:
        row = {
            "level": getattr(issue, "level", None),
            "code": getattr(issue, "code", None),
            "message": getattr(issue, "message", None),
            "field": getattr(issue, "field", None),
            "source_field": getattr(issue, "source_field", None),
            "row_count": getattr(issue, "row_count", None),
            "details": getattr(issue, "details", None),
        }
        rows.append(row)

    return pd.DataFrame(rows)

In [6]:
def make_field(name, dtype, *, required=False, constraints=None, domain=None):
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )

fields = {
    # Required
    "movement_id": make_field("movement_id", "string", required=True, constraints={"length": {"min": 1}, "unique": {"value": True}}),
    "user_id": make_field("user_id", "string", required=True, constraints={"length": {"min": 1}}),
    "origin_longitude": make_field("origin_longitude", "float", required=True, constraints={"range": {"min": -180.0, "max": 180.0}}),
    "origin_latitude": make_field("origin_latitude", "float", required=True, constraints={"range": {"min": -90.0, "max": 90.0}}),
    "destination_longitude": make_field("destination_longitude", "float", required=True, constraints={"range": {"min": -180.0, "max": 180.0}}),
    "destination_latitude": make_field("destination_latitude", "float", required=True, constraints={"range": {"min": -90.0, "max": 90.0}}),
    "origin_h3_index": make_field("origin_h3_index", "string", required=True, constraints={"h3": {"require_valid": True, "resolution": 8, "allow_mixed_resolution": False}}),
    "destination_h3_index": make_field("destination_h3_index", "string", required=True, constraints={"h3": {"require_valid": True, "resolution": 8, "allow_mixed_resolution": False}}),
    "origin_time_utc": make_field("origin_time_utc", "datetime", required=True),
    "destination_time_utc": make_field("destination_time_utc", "datetime", required=True),
    "trip_id": make_field("trip_id", "string", required=True, constraints={"length": {"min": 1}}),
    "movement_seq": make_field("movement_seq", "int", required=True, constraints={"range": {"min": 0, "max": 10}}),

    # Base / optional
    "origin_municipality": make_field("origin_municipality", "string"),
    "destination_municipality": make_field("destination_municipality", "string"),
    "timezone_offset_min": make_field("timezone_offset_min", "int", constraints={"range": {"min": -720, "max": 840}}),
    "origin_time_local_hhmm": make_field("origin_time_local_hhmm", "string"),
    "destination_time_local_hhmm": make_field("destination_time_local_hhmm", "string"),
    "trip_weight": make_field("trip_weight", "float", constraints={"range": {"min": 0.0, "max": 100000.0}}),
    "mode_sequence": make_field("mode_sequence", "string"),

    "mode": make_field("mode", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["mode"], extendable=True)),
    "purpose": make_field("purpose", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["purpose"], extendable=True)),
    "day_type": make_field("day_type", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["day_type"], extendable=True)),
    "time_period": make_field("time_period", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["time_period"], extendable=True)),
    "user_gender": make_field("user_gender", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["user_gender"], extendable=True)),
    "user_age_group": make_field("user_age_group", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["user_age_group"], extendable=True)),
    "income_quintile": make_field("income_quintile", "categorical", domain=DomainSpec(values=CANONICAL_DOMAINS["income_quintile"], extendable=True)),

    # Extra categorical fields
    "activity_destination": make_field("activity_destination", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["activity_destination"], extendable=True)),
    "season": make_field("season", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["season"], extendable=True)),
    "travel_time_quality_code": make_field("travel_time_quality_code", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["travel_time_quality_code"], extendable=True)),
    "fare_payment_type": make_field("fare_payment_type", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["fare_payment_type"], extendable=True)),
    "highway_avoid_reason": make_field("highway_avoid_reason", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["highway_avoid_reason"], extendable=True)),
    "bike_lane_usage": make_field("bike_lane_usage", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["bike_lane_usage"], extendable=True)),
    "bike_lane_issue": make_field("bike_lane_issue", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["bike_lane_issue"], extendable=True)),
    "bike_parking_type": make_field("bike_parking_type", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["bike_parking_type"], extendable=True)),
    "cut_type_stage_trip": make_field("cut_type_stage_trip", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["cut_type_stage_trip"], extendable=True)),
    "travel_time_bucket": make_field("travel_time_bucket", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["travel_time_bucket"], extendable=True)),
    "home_tenure": make_field("home_tenure", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["home_tenure"], extendable=True)),
    "education_level": make_field("education_level", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["education_level"], extendable=True)),
    "activity_status": make_field("activity_status", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["activity_status"], extendable=True)),
    "occupation_type": make_field("occupation_type", "categorical", domain=DomainSpec(values=EXTRA_CATEGORICAL_DOMAINS["occupation_type"], extendable=True)),

    # Extra numeric / string fields
    "travel_time_min": make_field("travel_time_min", "float", constraints={"range": {"min": 0.0, "max": 1000.0}}),
    "fare_amount": make_field("fare_amount", "float", constraints={"range": {"min": 0.0, "max": 100000.0}}),
    "distance_route_m": make_field("distance_route_m", "float", constraints={"range": {"min": 0.0, "max": 1000000.0}}),
    "in_vehicle_travel_time_min": make_field("in_vehicle_travel_time_min", "float", constraints={"range": {"min": 0.0, "max": 1000.0}}),
    "household_id": make_field("household_id", "string"),
    "public_transport_route_code": make_field("public_transport_route_code", "string"),
}

trip_schema = TripSchema(
    version="1.1-demo-happy-path",
    fields=fields,
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "origin_time_utc",
        "destination_time_utc",
        "trip_id",
        "movement_seq",
    ],
    semantic_rules=None,
)

print("Schema version:", trip_schema.version)
print("N campos schema:", len(trip_schema.fields))
print("N required:", len(trip_schema.required))

Schema version: 1.1-demo-happy-path
N campos schema: 46
N required: 12


## 3. Importación al formato Golondrina

Ahora sí voy a correr el import.

La expectativa en este punto es:
- corregir nombres de columnas vía `field_correspondence`,
- corregir valores fuente en `mode` y `purpose` vía `value_correspondence`,
- coercionar tipos mínimos,
- y obtener un `TripDataset` construible, trazable y listo para validación formal.

In [10]:
import_options = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone=None,
)

provenance = {
    "source": {
        "name": "synthetic_demo",
        "entity": "trips",
        "version": "2026-04-demo",
    },
    "ingestion": {
        "created_at_utc": "2026-04-04T00:00:00Z",
        "pipeline_id": "demo-happy-path-trips",
    },
    "input_fingerprint": {
        "rows": int(len(raw_df)),
        "columns": list(raw_df.columns),
        "uri": str(RAW_CSV_PATH),
    },
    "notes": [
        "Demo end-to-end de trips en happy path",
        "Dataset sintético rico con varias columnas categóricas",
    ],
}

trips, import_report = import_trips_from_dataframe(
    raw_df,
    trip_schema,
    source_name="synthetic_demo",
    options=import_options,
    field_correspondence=SOURCE_FIELD_CORRESPONDENCE,
    value_correspondence=VALUE_CORRESPONDENCE,
    provenance=provenance,
    h3_resolution=8,
)

print("Import ok:", import_report.ok)
display(import_report.summary)
display(issues_to_frame(import_report.issues).head(20))

print("is_validated después de import:", trips.metadata.get("is_validated"))
print("dataset_id:", trips.metadata.get("dataset_id"))
print("shape trips.data:", trips.data.shape)

display(trips.data.head())
#display(trips.data.dtypes.astype(str).to_frame("dtype").head(40))

Import ok: True


{'rows_in': 10000,
 'rows_out': 10000,
 'n_fields_mapped': 26,
 'n_domain_mappings_applied': 27}

,level,code,message,field,source_field,row_count,details
0,info,IMP.METADATA.DATASET_ID_CREATED,Se generó dataset_id para el dataset importado...,None,None,None,{'dataset_id': 'tripds_333a87e1087b44249ad0761...


is_validated después de import: False
dataset_id: tripds_333a87e1087b44249ad07611966f93cf
shape trips.data: (10000, 46)


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_destination,season,travel_time_quality_code,fare_payment_type,highway_avoid_reason,bike_lane_usage,bike_lane_issue,bike_parking_type,cut_type_stage_trip,travel_time_bucket,home_tenure,education_level,activity_status,occupation_type,travel_time_min,fare_amount,distance_route_m,in_vehicle_travel_time_min,household_id,public_transport_route_code
0,m00000,u1079,tm_00000,0,-70.594233,-33.452177,-70.546872,-33.444157,2026-03-08 01:48:00+00:00,2026-03-08 02:05:00+00:00,88b2c50b67fffff,88b2c50b51fffff,Vitacura,Vitacura,-180,01:48,02:05,4.488,train,bus,leisure,holiday,afternoon,male,unknown,unknown,recreación,winter,imputed,cash,no_need,not_applicable,missing_segments,private_building,cut_at_missing_stage,0-10,owned,secondary,retired,employer,17.0,1520.0,10653.7,17.0,H03082,I09c
1,m00001,u1079,tm_00000,1,-70.443241,-33.252172,-70.388813,-33.256286,2026-03-07 09:23:00+00:00,2026-03-07 10:53:00+00:00,88b2c5c4d7fffff,88b2c51315fffff,Independencia,Puente Alto,-180,09:23,10:53,2.493,bicycle,bus,leisure,weekend,morning,male,65-plus,unknown,salud,winter,missing_end,cash,too_expensive,never,unsafe,private_building,single_stage_only,41-60,owned,technical,retired,employee,90.0,1520.0,1357.5,90.0,H04605,L4
2,m00002,u1637,tm_00001,0,-70.681063,-33.341502,-70.705876,-33.341132,2026-03-05 04:40:00+00:00,2026-03-05 05:28:00+00:00,88b2c5cd8bfffff,88b2c5cda9fffff,Ñuñoa,Santiago,-180,04:40,05:28,3.416,bus,train,education,weekday,night,unknown,15-24,2,estudio,vacation,missing_end,card,too_expensive,not_applicable,missing_segments,none,single_stage_only,41-60,rented,technical,other,employee,48.0,800.0,1707.4,48.0,H04806,I09c
3,m00003,u1637,tm_00001,1,-70.557340,-33.481444,-70.565914,-33.449085,2026-03-04 10:41:00+00:00,2026-03-04 12:19:00+00:00,88b2c5086dfffff,88b2c50b09fffff,Pudahuel,Providencia,-180,10:41,12:19,4.030,car+car+car,bus,work,weekday,midday,male,15-24,3,recreación,vacation,reported_approx,card,no_need,not_applicable,crowded,none,cut_at_missing_stage,0-10,other,primary,studying,employee,98.0,800.0,3429.2,98.0,H03436,B07
4,m00004,u1637,tm_00001,2,-70.695597,-33.513595,-70.658855,-33.509041,2026-03-03 13:22:00+00:00,2026-03-03 14:08:00+00:00,88b2c54665fffff,88b2c54609fffff,Independencia,La Florida,-180,13:22,14:08,1.746,car+walk,car,leisure,weekday,afternoon,female,0-14,3,estudio,vacation,missing_end,integrated_fare,prefers_local_roads,always,missing_segments,private_building,cut_at_missing_stage,0-10,loaned,primary,retired,employer,46.0,1200.0,8659.5,46.0,H04054,401


### Exploracion del resultado del import

In [11]:
print("Campos presentes en trips.data:", len(trips.data.columns))
print("Primeros eventos en metadata:")
display(trips.metadata.get("events", [])[:1])

print("Schema effective:")
display(trips.schema_effective.to_dict())

print("Domains effective (muestra):")
display({
    k: trips.metadata["domains_effective"][k]
    for k in ["mode", "purpose", "day_type", "time_period", "user_gender"]
    if k in trips.metadata.get("domains_effective", {})
})

print("Valores ya normalizados de mode:")
print(sorted(trips.data["mode"].dropna().astype(str).unique().tolist()))

print("Valores ya normalizados de purpose:")
print(sorted(trips.data["purpose"].dropna().astype(str).unique().tolist()))

Campos presentes en trips.data: 46
Primeros eventos en metadata:


[{'op': 'import_trips',
  'ts_utc': '2026-04-04T21:49:43.327901+00:00',
  'parameters': {'keep_extra_fields': False,
   'selected_fields': None,
   'strict': False,
   'strict_domains': False,
   'single_stage': False,
   'h3_resolution': 8,
   'source_name': 'synthetic_demo',
   'source_timezone': None},
  'summary': {'input_rows': 10000,
   'output_rows': 10000,
   'rows_dropped': 0,
   'n_fields_mapped': 26,
   'n_domain_mappings_applied': 27,
   'columns_added': [],
   'columns_deleted': [],
   'domains_extended_count': 0,
   'temporal_tier': 'tier_1',
   'temporal_notes': None},
  'issues_summary': {'counts': {}, 'by_code': {}}}]

Schema effective:


{'dtype_effective': {'movement_id': 'string',
  'user_id': 'string',
  'origin_longitude': 'float',
  'origin_latitude': 'float',
  'destination_longitude': 'float',
  'destination_latitude': 'float',
  'origin_h3_index': 'string',
  'destination_h3_index': 'string',
  'origin_time_utc': 'datetime',
  'destination_time_utc': 'datetime',
  'trip_id': 'string',
  'movement_seq': 'int',
  'origin_municipality': 'string',
  'destination_municipality': 'string',
  'timezone_offset_min': 'int',
  'origin_time_local_hhmm': 'string',
  'destination_time_local_hhmm': 'string',
  'trip_weight': 'float',
  'mode_sequence': 'string',
  'mode': 'categorical',
  'purpose': 'categorical',
  'day_type': 'categorical',
  'time_period': 'categorical',
  'user_gender': 'categorical',
  'user_age_group': 'categorical',
  'income_quintile': 'categorical',
  'activity_destination': 'categorical',
  'season': 'categorical',
  'travel_time_quality_code': 'categorical',
  'fare_payment_type': 'categorical',
  

Domains effective (muestra):


{'mode': {'base_values': ['bicycle',
   'bus',
   'car',
   'metro',
   'motorcycle',
   'other',
   'ride_hailing',
   'scooter',
   'taxi',
   'train',
   'walk'],
  'observed_values': ['bicycle',
   'bus',
   'car',
   'metro',
   'motorcycle',
   'other',
   'ride_hailing',
   'scooter',
   'taxi',
   'train',
   'walk'],
  'extended_values': [],
  'unknown_values': [],
  'extendable': True,
  'unknown_value': 'other',
  'n_unique_observed': 11,
  'n_added': 0,
  'value_correspondence_applied': {'Auto Chofer': 'car',
   'Auto Acompañante': 'car',
   'Bus alimentador': 'bus',
   'Bus troncal': 'bus',
   'Bus institucional': 'bus',
   'Bus interurbano o rural': 'bus',
   'Bus urbano pago conductor': 'bus',
   'Taxi colectivo': 'taxi',
   'Taxi o radiotaxi': 'taxi',
   'Furgón escolar pasajero': 'bus',
   'Furgón escolar chofer': 'bus',
   'Servicio informal': 'other',
   'METROTREN': 'train',
   'ZP': 'bus'},
  'values': ['bicycle',
   'bus',
   'car',
   'metro',
   'motorcycle',
  

Valores ya normalizados de mode:
['bicycle', 'bus', 'car', 'metro', 'motorcycle', 'other', 'ride_hailing', 'scooter', 'taxi', 'train', 'walk']
Valores ya normalizados de purpose:
['education', 'errand', 'health', 'home', 'leisure', 'other', 'shopping', 'transfer', 'work']


## 4. Validación formal inicial

Hasta aquí el import dejó el dataset construido y estandarizado, pero no validado formalmente.

Ahora voy a ejecutar una validación más exigente:
- required fields,
- types/formats,
- constraints,
- dominios completos,
- consistencia temporal,
- consistencia cross-field,
- y duplicados.

In [14]:
validation_options = ValidationOptions(
    strict=False,
    max_issues=200,
    sample_rows_per_issue=5,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    domains_sample_frac=0.01,
    domains_min_in_domain_ratio=1.0,
    validate_temporal_consistency=True,
    validate_duplicates=True,
    duplicates_subset=("movement_id",),
)

validation_report_1 = validate_trips(
    trips,
    options=validation_options,
)

print("Validation ok:", validation_report_1.ok)
display(validation_report_1.summary)
display(issues_to_frame(validation_report_1.issues).head(20))

print("is_validated después de validate:", trips.metadata.get("is_validated"))
print("Último evento:", trips.metadata["events"][-1]["op"])

Validation ok: True


{'ok': True,
 'n_rows': 10000,
 'n_issues': 1,
 'n_errors': 0,
 'n_warnings': 1,
 'n_info': 0,
 'counts_by_level': {'error': 0, 'warning': 1, 'info': 0},
 'counts_by_code': {'VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID': 1},
 'checked_fields': ['activity_destination',
  'activity_status',
  'bike_lane_issue',
  'bike_lane_usage',
  'bike_parking_type',
  'cut_type_stage_trip',
  'day_type',
  'destination_h3_index',
  'destination_latitude',
  'destination_longitude',
  'destination_municipality',
  'destination_time_local_hhmm',
  'destination_time_utc',
  'distance_route_m',
  'education_level',
  'fare_amount',
  'fare_payment_type',
  'highway_avoid_reason',
  'home_tenure',
  'household_id',
  'in_vehicle_travel_time_min',
  'income_quintile',
  'mode',
  'mode_sequence',
  'movement_id',
  'movement_seq',
  'occupation_type',
  'origin_h3_index',
  'origin_latitude',
  'origin_longitude',
  'origin_municipality',
  'origin_time_local_hhmm',
  'origin_time_utc',
  'public_transport_route

,level,code,message,field,source_field,row_count,details
0,warning,VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID,La constraint 'unique' del campo 'movement_id'...,movement_id,None,None,"{'check': 'constraints', 'field': 'movement_id..."


is_validated después de validate: True
Último evento: validate_trips


## 5. Persistencia formal en formato Golondrina

Ahora voy a escribir el dataset validado a disco con `write_trips`.

Aquí quiero observar dos cosas:
1. que se materialice correctamente el bundle formal,
2. y que el parquet realmente muestre una ventaja de tamaño frente al CSV crudo.

In [15]:
write_options = WriteTripsOptions(
    mode="overwrite",
    require_validated=True,
    storage_format="parquet",
    parquet_compression="snappy",
    normalize_artifact_dir=True,
)

write_report = write_trips(
    trips,
    ARTIFACT_BASE_PATH,
    options=write_options,
)

print("Write ok:", write_report.ok)
display(write_report.summary)
display(pd.DataFrame([issue.to_dict() for issue in write_report.issues]).head(20) if write_report.issues else pd.DataFrame())

print("Artifact dir exists:", ARTIFACT_DIR.exists())
print("Files:", [p.name for p in ARTIFACT_DIR.iterdir()])

Write ok: True


{'n_rows': 10000,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\demo_trips_happy_path.golondrina',
 'dataset_id': 'tripds_333a87e1087b44249ad07611966f93cf',
 'artifact_id': 'art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

""


Artifact dir exists: True
Files: ['trips.metadata.json', 'trips.parquet']


### Inspección de los datos persistidos

In [16]:
sidecar_path = ARTIFACT_DIR / "trips.metadata.json"
parquet_path = ARTIFACT_DIR / "trips.parquet"

sidecar = json.loads(sidecar_path.read_text(encoding="utf-8"))

print("Sidecar keys:")
print(sidecar.keys())

print("dataset_type:", sidecar["dataset_type"])
print("format:", sidecar["format"])
print("layout_version:", sidecar["layout_version"])
print("storage:", sidecar["storage"])
print("dataset_id:", sidecar["dataset_id"])
print("artifact_id:", sidecar["artifact_id"])

print("Files block:")
display(sidecar["files"])

print("Metadata persistida - keys:")
print(sidecar["metadata"].keys())

Sidecar keys:
dict_keys(['dataset_type', 'format', 'layout_version', 'storage', 'dataset_id', 'artifact_id', 'files', 'schema', 'schema_effective', 'provenance', 'metadata'])
dataset_type: trips
format: golondrina
layout_version: 1.1
storage: {'format': 'parquet', 'options': {'compression': 'snappy'}}
dataset_id: tripds_333a87e1087b44249ad07611966f93cf
artifact_id: art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff
Files block:


{'data': 'trips.parquet', 'metadata': 'trips.metadata.json'}

Metadata persistida - keys:
dict_keys(['dataset_id', 'is_validated', 'schema', 'schema_effective', 'mappings', 'domains_effective', 'domains_extended', 'extra_fields_kept', 'events', 'provenance', 'temporal', 'h3', 'artifact_id'])


### Comparación de tamaños CSV vs Parquet

In [17]:
csv_size = RAW_CSV_PATH.stat().st_size
parquet_size = parquet_path.stat().st_size

size_df = pd.DataFrame(
    {
        "artifact": ["csv_crudo_fuente", "parquet_golondrina"],
        "bytes": [csv_size, parquet_size],
        "mb": [csv_size / (1024**2), parquet_size / (1024**2)],
    }
)
size_df["ratio_vs_csv"] = size_df["bytes"] / csv_size

display(size_df)
print(f"Reducción relativa parquet/csv: {parquet_size / csv_size:.3f}")

,artifact,bytes,mb,ratio_vs_csv
0,csv_crudo_fuente,4577737,4.365670,1.000000
1,parquet_golondrina,1035161,0.987206,0.226129


Reducción relativa parquet/csv: 0.226


### Verificación de dictionary encoding en columnas categóricas

In [18]:
parquet_file = pq.ParquetFile(parquet_path)
try:
    names = parquet_file.schema_arrow.names
    rows = []

    categorical_fields_to_check = [
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "user_gender",
        "user_age_group",
        "income_quintile",
        "season",
        "fare_payment_type",
        "bike_lane_usage",
        "travel_time_bucket",
        "education_level",
        "activity_status",
        "occupation_type",
    ]

    for col_name in categorical_fields_to_check:
        if col_name not in names:
            continue
        col_idx = names.index(col_name)
        encodings = {
            str(enc).upper()
            for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
        }
        rows.append(
            {
                "column": col_name,
                "encodings": sorted(encodings),
                "uses_dictionary": any("DICTIONARY" in enc for enc in encodings),
            }
        )

    display(pd.DataFrame(rows))
finally:
    parquet_file.close()

,column,encodings,uses_dictionary
0,mode,"[PLAIN, RLE, RLE_DICTIONARY]",True
1,purpose,"[PLAIN, RLE, RLE_DICTIONARY]",True
2,day_type,"[PLAIN, RLE, RLE_DICTIONARY]",True
3,time_period,"[PLAIN, RLE, RLE_DICTIONARY]",True
4,user_gender,"[PLAIN, RLE, RLE_DICTIONARY]",True
5,user_age_group,"[PLAIN, RLE, RLE_DICTIONARY]",True
6,income_quintile,"[PLAIN, RLE, RLE_DICTIONARY]",True
7,season,"[PLAIN, RLE, RLE_DICTIONARY]",True
8,fare_payment_type,"[PLAIN, RLE, RLE_DICTIONARY]",True
9,bike_lane_usage,"[PLAIN, RLE, RLE_DICTIONARY]",True


## 6. Lectura formal con `read_trips`

Ahora voy a reconstruir el dataset desde el bundle persistido.

Por diseño de la operación, aquí espero dos cosas:
- que el dataset se reconstruya correctamente,
- y que `metadata["is_validated"]` vuelva a `False`, porque la lectura formal no equivale a una certificación de conformidad.

In [20]:
read_options = ReadTripsOptions(
    schema=None,
    strict=False,
    keep_metadata=True,
)

trips_loaded, read_report = read_trips(
    ARTIFACT_BASE_PATH,
    options=read_options,
)

print("Read ok:", read_report.ok)
display(read_report.summary)
display(issues_to_frame(read_report.issues).head(20))

print("is_validated después de read:", trips_loaded.metadata.get("is_validated"))
print("dataset_id:", trips_loaded.metadata.get("dataset_id"))
print("artifact_id:", trips_loaded.metadata.get("artifact_id"))
print("shape trips_loaded.data:", trips_loaded.data.shape)

display(trips_loaded.data.head())

Read ok: True


{'n_rows': 10000,
 'n_columns': 46,
 'path': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\demo_trips_happy_path.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_333a87e1087b44249ad07611966f93cf',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff',
 'artifact_id_status': 'loaded'}

,level,code,message,field,source_field,row_count,details
0,info,READ.METADATA.VALIDATED_FORCED_FALSE,Se forzó metadata['is_validated']=False tras l...,None,None,None,"{'previous_value': True, 'new_value': False, '..."


is_validated después de read: False
dataset_id: tripds_333a87e1087b44249ad07611966f93cf
artifact_id: art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff
shape trips_loaded.data: (10000, 46)


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_destination,season,travel_time_quality_code,fare_payment_type,highway_avoid_reason,bike_lane_usage,bike_lane_issue,bike_parking_type,cut_type_stage_trip,travel_time_bucket,home_tenure,education_level,activity_status,occupation_type,travel_time_min,fare_amount,distance_route_m,in_vehicle_travel_time_min,household_id,public_transport_route_code
0,m00000,u1079,tm_00000,0,-70.594233,-33.452177,-70.546872,-33.444157,2026-03-08 01:48:00+00:00,2026-03-08 02:05:00+00:00,88b2c50b67fffff,88b2c50b51fffff,Vitacura,Vitacura,-180,01:48,02:05,4.488,train,bus,leisure,holiday,afternoon,male,unknown,unknown,recreación,winter,imputed,cash,no_need,not_applicable,missing_segments,private_building,cut_at_missing_stage,0-10,owned,secondary,retired,employer,17.0,1520.0,10653.7,17.0,H03082,I09c
1,m00001,u1079,tm_00000,1,-70.443241,-33.252172,-70.388813,-33.256286,2026-03-07 09:23:00+00:00,2026-03-07 10:53:00+00:00,88b2c5c4d7fffff,88b2c51315fffff,Independencia,Puente Alto,-180,09:23,10:53,2.493,bicycle,bus,leisure,weekend,morning,male,65-plus,unknown,salud,winter,missing_end,cash,too_expensive,never,unsafe,private_building,single_stage_only,41-60,owned,technical,retired,employee,90.0,1520.0,1357.5,90.0,H04605,L4
2,m00002,u1637,tm_00001,0,-70.681063,-33.341502,-70.705876,-33.341132,2026-03-05 04:40:00+00:00,2026-03-05 05:28:00+00:00,88b2c5cd8bfffff,88b2c5cda9fffff,Ñuñoa,Santiago,-180,04:40,05:28,3.416,bus,train,education,weekday,night,unknown,15-24,2,estudio,vacation,missing_end,card,too_expensive,not_applicable,missing_segments,none,single_stage_only,41-60,rented,technical,other,employee,48.0,800.0,1707.4,48.0,H04806,I09c
3,m00003,u1637,tm_00001,1,-70.557340,-33.481444,-70.565914,-33.449085,2026-03-04 10:41:00+00:00,2026-03-04 12:19:00+00:00,88b2c5086dfffff,88b2c50b09fffff,Pudahuel,Providencia,-180,10:41,12:19,4.030,car+car+car,bus,work,weekday,midday,male,15-24,3,recreación,vacation,reported_approx,card,no_need,not_applicable,crowded,none,cut_at_missing_stage,0-10,other,primary,studying,employee,98.0,800.0,3429.2,98.0,H03436,B07
4,m00004,u1637,tm_00001,2,-70.695597,-33.513595,-70.658855,-33.509041,2026-03-03 13:22:00+00:00,2026-03-03 14:08:00+00:00,88b2c54665fffff,88b2c54609fffff,Independencia,La Florida,-180,13:22,14:08,1.746,car+walk,car,leisure,weekday,afternoon,female,0-14,3,estudio,vacation,missing_end,integrated_fare,prefers_local_roads,always,missing_segments,private_building,cut_at_missing_stage,0-10,loaned,primary,retired,employer,46.0,1200.0,8659.5,46.0,H04054,401


In [21]:
print("Operaciones registradas en metadata['events']:")
print([ev["op"] for ev in trips_loaded.metadata.get("events", [])])

print("Comparación rápida de shapes:")
print("trips original:", trips.data.shape)
print("trips loaded  :", trips_loaded.data.shape)

print("Schema original vs cargado:")
print(trips.schema.version, " | ", trips_loaded.schema.version)

print("Schema effective cargado:")
display(trips_loaded.schema_effective.to_dict())

Operaciones registradas en metadata['events']:
['import_trips', 'validate_trips', 'validate_trips', 'write_trips', 'read_trips']
Comparación rápida de shapes:
trips original: (10000, 46)
trips loaded  : (10000, 46)
Schema original vs cargado:
1.1-demo-happy-path  |  1.1-demo-happy-path
Schema effective cargado:


{'dtype_effective': {'movement_id': 'string',
  'user_id': 'string',
  'origin_longitude': 'float',
  'origin_latitude': 'float',
  'destination_longitude': 'float',
  'destination_latitude': 'float',
  'origin_h3_index': 'string',
  'destination_h3_index': 'string',
  'origin_time_utc': 'datetime',
  'destination_time_utc': 'datetime',
  'trip_id': 'string',
  'movement_seq': 'int',
  'origin_municipality': 'string',
  'destination_municipality': 'string',
  'timezone_offset_min': 'int',
  'origin_time_local_hhmm': 'string',
  'destination_time_local_hhmm': 'string',
  'trip_weight': 'float',
  'mode_sequence': 'string',
  'mode': 'categorical',
  'purpose': 'categorical',
  'day_type': 'categorical',
  'time_period': 'categorical',
  'user_gender': 'categorical',
  'user_age_group': 'categorical',
  'income_quintile': 'categorical',
  'activity_destination': 'categorical',
  'season': 'categorical',
  'travel_time_quality_code': 'categorical',
  'fare_payment_type': 'categorical',
  

## 7. Revalidación del dataset cargado

Este último paso demuestra el comportamiento esperado del bloque de persistencia:
- después de `read_trips`, el dataset queda no validado,
- pero si el artefacto formal era consistente, entonces puede validarse nuevamente sin problemas.

In [23]:
validation_report_2 = validate_trips(
    trips_loaded,
    options=validation_options,
)

print("Validation 2 ok:", validation_report_2.ok)
display(validation_report_2.summary)
display(issues_to_frame(validation_report_2.issues).head(20))

print("is_validated después de revalidar:", trips_loaded.metadata.get("is_validated"))
print("Último evento:", trips_loaded.metadata["events"][-1]["op"])

Validation 2 ok: True


{'ok': True,
 'n_rows': 10000,
 'n_issues': 1,
 'n_errors': 0,
 'n_warnings': 1,
 'n_info': 0,
 'counts_by_level': {'error': 0, 'warning': 1, 'info': 0},
 'counts_by_code': {'VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID': 1},
 'checked_fields': ['activity_destination',
  'activity_status',
  'bike_lane_issue',
  'bike_lane_usage',
  'bike_parking_type',
  'cut_type_stage_trip',
  'day_type',
  'destination_h3_index',
  'destination_latitude',
  'destination_longitude',
  'destination_municipality',
  'destination_time_local_hhmm',
  'destination_time_utc',
  'distance_route_m',
  'education_level',
  'fare_amount',
  'fare_payment_type',
  'highway_avoid_reason',
  'home_tenure',
  'household_id',
  'in_vehicle_travel_time_min',
  'income_quintile',
  'mode',
  'mode_sequence',
  'movement_id',
  'movement_seq',
  'occupation_type',
  'origin_h3_index',
  'origin_latitude',
  'origin_longitude',
  'origin_municipality',
  'origin_time_local_hhmm',
  'origin_time_utc',
  'public_transport_route

,level,code,message,field,source_field,row_count,details
0,warning,VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID,La constraint 'unique' del campo 'movement_id'...,movement_id,None,None,"{'check': 'constraints', 'field': 'movement_id..."


is_validated después de revalidar: True
Último evento: validate_trips


In [24]:
print("Comparación final de dataframes (sin exigir identidad exacta de dtypes pandas):")
pd.testing.assert_frame_equal(
    trips.data.reset_index(drop=True),
    trips_loaded.data.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

print("OK - Los datos tabulares se reconstruyeron correctamente.")

print("dataset_id original :", trips.metadata["dataset_id"])
print("dataset_id leído    :", trips_loaded.metadata["dataset_id"])
print("artifact_id original:", trips.metadata["artifact_id"])
print("artifact_id leído   :", trips_loaded.metadata["artifact_id"])

print("Schema equal:", trips.schema.to_dict() == trips_loaded.schema.to_dict())
print("Schema effective equal:", trips.schema_effective.to_dict() == trips_loaded.schema_effective.to_dict())

Comparación final de dataframes (sin exigir identidad exacta de dtypes pandas):
OK - Los datos tabulares se reconstruyeron correctamente.
dataset_id original : tripds_333a87e1087b44249ad07611966f93cf
dataset_id leído    : tripds_333a87e1087b44249ad07611966f93cf
artifact_id original: art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff
artifact_id leído   : art_7f5fe32b-06fd-458f-a9a2-1c9e09249eff
Schema equal: True
Schema effective equal: True


## 8. Conclusión de la demo

En este happy path se observó lo siguiente:

1. El import logró construir un `TripDataset` Golondrina desde una fuente externa con nombres y valores no completamente canónicos.
2. La primera validación certificó formalmente el dataset.
3. `write_trips` persistió correctamente el bundle formal Golondrina.
4. El `trips.parquet` resultó más eficiente que el CSV crudo y mostró evidencia de dictionary encoding en varias columnas categóricas.
5. `read_trips` reconstruyó correctamente el dataset, pero dejó `is_validated=False` por diseño.
6. Una segunda validación confirmó que el dataset reconstruido sigue siendo conforme al contrato.

Con esto queda demostrado el flujo básico completo de trips en el módulo.